# OpusClip Pipeline — Google Colab Demo

**OpusClip** is an AI-powered pipeline that transforms long-form videos (YouTube or local)
into short, publication-ready vertical clips with:
- **Smart Clip Selection** — LLM-based virality scoring
- **Face-Aware Cropping** — MediaPipe + SmartDirector crop logic
- **Bilingual Subtitles** — ASS-format karaoke for Arabic & English
- **GPU-Accelerated** — CUDA Whisper + NVENC encoding (when available)

---
### Runtime Setup
Before running any cells:
1. **Runtime -> Change runtime type -> T4 GPU** (recommended)
2. Set your **OPUSCLIP_API_KEY** in the Secrets panel (key icon) or use the cell below

---
## 1. Install Dependencies

Installs the `opusclip` package and required system binaries.
This cell typically runs in **2-3 minutes** on Colab. Safe to re-run.

In [ ]:
import subprocess, sys, os, pathlib, json, importlib

def _sh(cmd, label=""):
    print(f"  -> {label or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr.strip():
        print(f"    ! {r.stderr.strip()[:200]}")
    return r

# --- ffmpeg ---
ffmpeg_ok = _sh("ffmpeg -version", "ffmpeg").returncode == 0
if not ffmpeg_ok:
    print("  -> Installing ffmpeg ...")
    _sh("apt-get install -y -q ffmpeg > /dev/null 2>&1", "apt ffmpeg")
    ffmpeg_ok = _sh("ffmpeg -version", "verify ffmpeg").returncode == 0
    if not ffmpeg_ok:
        raise RuntimeError("ffmpeg installation failed")

# --- yt-dlp ---
if _sh("yt-dlp --version", "yt-dlp").returncode != 0:
    _sh("pip install -q yt-dlp", "pip yt-dlp")

# --- opusclip + its deps ---
_sh("pip install -q opusclip", "pip opusclip")

# --- fonts ---
_sh("apt-get install -y -q fonts-noto-core fonts-noto-extra "
    "fonts-dejavu-core > /dev/null 2>&1", "fonts")

# --- verify imports ---
imports = ["opusclip", "yt_dlp"]
for mod_name in imports:
    mod = importlib.import_module(mod_name)
    ver = getattr(mod, "__version__", None) or "ok"
    print(f"  {mod_name} == {ver}")

print("\nAll dependencies ready.")

---
## 2. Configuration

Set your **API key** and choose a video source.

> **Free API options:**
> - [opencode.ai](https://opencode.ai/zen) -- `deepseek-v4-flash-free` (default)
> - [Groq](https://console.groq.com) -- `llama-3.3-70b-versatile`
> - [Gemini](https://aistudio.google.com/apikey) -- `gemini-2.0-flash`

Store your API key in Colab's **Secrets** (key icon) panel as `OPUSCLIP_API_KEY`,
or paste it directly below.

This cell is safe to re-run.

In [ ]:
# --- API Key ---
try:
    from google.colab import userdata
    os.environ["OPUSCLIP_API_KEY"] = userdata.get("OPUSCLIP_API_KEY")
    print("API key loaded from Colab Secrets")
except (ImportError, ValueError, KeyError):
    os.environ.setdefault("OPUSCLIP_API_KEY", "")
    print("No Colab Secrets found. Paste your key below if not set.")

# --- LLM Configuration (only set if not already provided) ---
os.environ.setdefault("LLM_BASE_URL", "https://opencode.ai/zen/v1")
os.environ.setdefault("LLM_MODEL", "deepseek-v4-flash-free")

# --- Verify ---
if not os.environ.get("OPUSCLIP_API_KEY"):
    raise RuntimeError(
        "OPUSCLIP_API_KEY is not set. "
        "Add it to Colab Secrets or paste it in the cell above."
    )

print(f"LLM endpoint : {os.environ['LLM_BASE_URL']}")
print(f"LLM model    : {os.environ['LLM_MODEL']}")
print(f"API key      : ...{os.environ['OPUSCLIP_API_KEY'][-4:]} (last 4 chars)")

---
## 3. Choose Your Video Source

Set `INPUT_MODE` to `"youtube"` (paste a URL) or `"upload"` (pick a local file).

Safe to re-run. Changes to `INPUT_MODE` or `YOUTUBE_URL` take effect immediately.

In [ ]:
from opusclip.config import PipelineConfig

# --- Pick ONE mode ---
INPUT_MODE  = "youtube"   # "upload"  |  "youtube"
YOUTUBE_URL = "https://youtu.be/OwDU3GH9cGI?si=MeHrfxVtdIUK5u9c"  # paste URL if "youtube"

# --- Resolve video path ---
config = PipelineConfig.from_env()
work_dir = pathlib.Path.cwd()  # /content/ on Colab

VIDEO_PATH = None
if INPUT_MODE == "upload":
    from google.colab import files as _cf
    print("Select your video file ...")
    _uploaded = _cf.upload()
    VIDEO_PATH = str(work_dir / list(_uploaded.keys())[0])

elif INPUT_MODE == "youtube":
    if not YOUTUBE_URL:
        raise ValueError("Set YOUTUBE_URL above before continuing.")
    dest = work_dir / "source_video.mp4"
    VIDEO_PATH = str(dest)
    if dest.exists():
        print(f"Cached: {VIDEO_PATH} ({dest.stat().st_size / 1_048_576:.1f} MB)")
    else:
        print(f"Downloading: {YOUTUBE_URL}")
        r = _sh(
            'yt-dlp --format "bestvideo[height<=1080][ext=mp4]'
            '+bestaudio[ext=m4a]/best[height<=1080]" '
            '--merge-output-format mp4 --no-playlist '
            f'--output "{VIDEO_PATH}" "{YOUTUBE_URL}"',
        )
        if r.returncode != 0:
            raise RuntimeError(
                f"YouTube download failed (exit code {r.returncode}).\n"
                f"Stderr: {r.stderr[-300:]}"
            )
        if not dest.exists():
            raise RuntimeError(
                "Download completed but source_video.mp4 was not found."
            )
        print(f"Downloaded: {VIDEO_PATH} ({dest.stat().st_size / 1_048_576:.1f} MB)")

print(f"\nSource: {VIDEO_PATH}")

---
## 4. Run Pipeline

This cell uses `opusclip`'s production `ProviderFactory` and `Pipeline` classes.
All 10 pipeline steps run automatically:

| # | Step | Description |
|---|------|-------------|
| 1 | Validating input | Checks source URL or file path |
| 2 | Reading metadata | Extracts video info via ffprobe |
| 3 | Transcribing audio | faster-whisper (CUDA if available) |
| 4 | Repairing transcript | Fills missing words from Whisper |
| 5 | Selecting clips | LLM-based virality scoring |
| 6 | Rendering subtitles | ASS karaoke for Arabic/English |
| 7 | Rendering videos | FFmpeg single-pass + face crop |
| 8 | Validating outputs | ffprobe resolution/codec check |
| 9 | Generating metadata | LLM-based titles and hashtags |
| 10 | Producing outputs | Final summary JSON |

Safe to re-run after changing the source in the previous cell.

In [ ]:
from opusclip.provider_factory import ProviderFactory
from opusclip.exceptions import OpusClipError
import time

if not VIDEO_PATH:
    raise RuntimeError("No VIDEO_PATH set. Run the source selection cell first.")

config = PipelineConfig.from_env()
print(f"Output dir : {config.output_dir}")
print(f"LLM model  : {config.llm_model}")
print(f"Whisper    : {config.whisper_model} ({config.whisper_device})")
print(f"Encoder    : {config.encoder}")
print(f"Renderer   : {config.renderer_backend}")
print()

factory = ProviderFactory(config)
pipeline = factory.create_pipeline(VIDEO_PATH)

print("Starting pipeline ...\n")
t0 = time.time()

try:
    result = pipeline.run(VIDEO_PATH)
    elapsed = time.time() - t0
    print(f"\nPipeline finished in {elapsed:.1f}s")
except OpusClipError as e:
    elapsed = time.time() - t0
    print(f"\nPipeline failed after {elapsed:.1f}s: {e}")
    result = None

---
## 5. Results

Preview the pipeline output: generated clips, metadata, and performance metrics.

This cell is **guarded** -- it only runs if the pipeline completed successfully.

In [ ]:
if result is None:
    print("Pipeline did not complete. Nothing to show.")
else:
    print(f"Source duration : {result.duration:.1f}s ({result.duration/60:.1f} min)")
    print(f"Total clips     : {result.total_clips}")
    print(f"Successful      : {result.successful_clips}")
    print(f"Failed          : {result.failed_clips}")
    print()

    if result.total_clips == 0:
        print("No clips were produced. Check the pipeline logs above.")
    else:
        for clip in result.clips:
            if clip.success:
                print(f"Clip #{clip.number}")
                if clip.video_path and clip.video_path.exists():
                    sz = clip.video_path.stat().st_size / 1_048_576
                    print(f"  Video : {clip.video_path.name} ({sz:.1f} MB)")
                if clip.thumbnail_path and clip.thumbnail_path.exists():
                    print(f"  Thumb : {clip.thumbnail_path.name}")
                if clip.metadata:
                    print(f"  Title : {clip.metadata.title}")
            else:
                print(f"Clip #{clip.number} FAILED")
                if clip.error:
                    print(f"  Error: {clip.error}")
            print()

    print("Pipeline Metrics:")
    print(pipeline.metrics.report())

---
## 6. Download Results

Download individual clips or the entire output folder as a ZIP archive.

This cell is **guarded** -- it only runs if the pipeline completed successfully
and at least one clip was produced.

In [ ]:
if result is None or result.total_clips == 0:
    print("No clips to download.")
    print(f"Output directory (if any): {config.output_dir.resolve()}")
else:
    import zipfile

    OUTPUT_DIR = str(config.output_dir)
    abs_dir = pathlib.Path(OUTPUT_DIR).resolve()
    print(f"Output directory: {abs_dir}")

    zip_path = "/content/opusclip_results.zip"
    if abs_dir.exists() and any(abs_dir.iterdir()):
        print(f"Creating {zip_path} ...")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in abs_dir.rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(abs_dir))
        print(f"  Created ({abs_dir.stat().st_size / 1_048_576:.1f} MB bundle)")

    try:
        from google.colab import files
        if abs_dir.exists() and any(abs_dir.iterdir()):
            print("\nClick to download bundle:")
            files.download(zip_path)
            print("\nIndividual successful clips:")
            for clip in result.clips:
                if clip.success and clip.video_path and clip.video_path.exists():
                    files.download(str(clip.video_path))
    except ImportError:
        print(f"\nOutput files are in: {abs_dir}")

---
## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| `OPUSCLIP_API_KEY is not set` | Missing API key | Set it in Colab Secrets or paste in Cell 2 |
| `No module named 'opusclip'` | Package not installed | **Runtime -> Restart and run all** |
| `ffmpeg not found` | Missing binary | Re-run Cell 1 |
| `CUDA out of memory` | GPU OOM | Set `WHISPER_MODEL=medium` or `small` in Cell 2 |
| Pipeline too slow | No GPU selected | **Runtime -> Change runtime type -> T4 GPU** |
| YouTube download fails | Region/age restriction | Try a different URL or use `"upload"` mode |
| Blank subtitles | Missing fonts | Re-run Cell 1 (font install step) |

---
*For more details, see the [OpusClip documentation](https://github.com/ABo-EsMaiL/OpusClip).*